# 05 — Modeling: Functional Connectivity (Power-264)

Compares multiple classifiers for MDD vs. HC using FC features.  
Feature selection (`SelectKBest`) is applied **inside** the cross-validation loop  
to prevent data leakage.

**Input:** `data/processed/datasetConnectivityLabels.csv`  
**Output:** `results/metrics/fc_cv_results.csv` + figures

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay

RESULTS_DIR = "../results"
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/metrics",  exist_ok=True)

In [ ]:
df = pd.read_csv("../data/processed/datasetConnectivityLabels.csv")

fc_cols = [c for c in df.columns if c.startswith("fc_")]
mask = df["diagnosis"].notna()

X = df.loc[mask, fc_cols].values
y = df.loc[mask, "diagnosis"].astype(int).values

print(f"Subjects  : {X.shape[0]}")
print(f"Features  : {X.shape[1]:,}")
print(f"MDD / HC  : {y.sum()} / {(y==0).sum()}")

## Data leakage note!

Feature selection **must** be inside the CV pipeline so the F-scores  
are computed only on training folds — never on the held-out test fold.  
Placing `SelectKBest` before `train_test_split` or outside `cross_validate`  
leaks label information and inflates reported accuracy.

In [ ]:
K_FEATURES = 1000   # top-k features selected per fold

selector = lambda: SelectKBest(score_func=f_classif, k=K_FEATURES)

models = {
    "SVM (RBF)": Pipeline([
        ("scaler",   StandardScaler()),
        ("selector", selector()),
        ("clf",      SVC(kernel="rbf", C=10, gamma="scale",
                         class_weight="balanced", probability=True, random_state=42))
    ]),
    "Logistic Regression": Pipeline([
        ("scaler",   StandardScaler()),
        ("selector", selector()),
        ("clf",      LogisticRegression(class_weight="balanced",
                                        max_iter=1000, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("selector", selector()),
        ("clf",      RandomForestClassifier(n_estimators=200,
                                             class_weight="balanced", random_state=42))
    ]),
    "Gradient Boosting": Pipeline([
        ("selector", selector()),
        ("clf",      GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                                 max_depth=3, random_state=42))
    ]),
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = ["accuracy", "roc_auc", "f1", "precision", "recall"]

rows = []
for name, pipe in models.items():
    print(f"Running {name}...")
    scores = cross_validate(pipe, X, y, cv=cv,
                            scoring=scoring, n_jobs=-1)
    rows.append({
        "Model":          name,
        "Accuracy":       np.mean(scores["test_accuracy"]),
        "Accuracy_std":   np.std( scores["test_accuracy"]),
        "ROC_AUC":        np.mean(scores["test_roc_auc"]),
        "ROC_AUC_std":    np.std( scores["test_roc_auc"]),
        "F1":             np.mean(scores["test_f1"]),
        "Precision":      np.mean(scores["test_precision"]),
        "Recall":         np.mean(scores["test_recall"]),
    })

results_df = pd.DataFrame(rows).sort_values("ROC_AUC", ascending=False)
results_df.to_csv(f"{RESULTS_DIR}/metrics/fc_cv_results.csv", index=False)
results_df.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
metrics = ["Accuracy", "ROC_AUC", "F1", "Precision", "Recall"]
x = np.arange(len(results_df))
width = 0.15

for i, m in enumerate(metrics):
    ax.bar(x + i * width, results_df[m], width, label=m)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(results_df["Model"], rotation=15, ha="right")
ax.set_ylim(0.4, 0.85)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="Chance")
ax.set_title("5-Fold CV — FC Power-264 features")
ax.legend(bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/figures/fc_model_comparison.png", dpi=150)
plt.show()

## Best model — detailed evaluation

In [ ]:
from sklearn.model_selection import train_test_split

best_name = results_df.iloc[0]["Model"]
best_pipe = models[best_name]
print(f"Best model: {best_name}")

# Single hold-out split for confusion matrix and ROC curve
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
best_pipe.fit(X_train, y_train)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ConfusionMatrixDisplay.from_estimator(
    best_pipe, X_test, y_test,
    display_labels=["HC", "MDD"],
    ax=axes[0], colorbar=False
)
axes[0].set_title(f"Confusion Matrix — {best_name}")

RocCurveDisplay.from_estimator(
    best_pipe, X_test, y_test, ax=axes[1]
)
axes[1].set_title(f"ROC Curve — {best_name}")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/figures/fc_best_model_eval.png", dpi=150)
plt.show()